In [ ]:
import webbrowser                                           # To open Gmail automatically in the browser
import time                                                 # To manage delays during browser loading
import pandas as pd                                         # For data handling and table display
import gradio as gr                                         # To create the user interface (UI)
import mysql.connector                                      # To connect to the MySQL database
import urllib.parse                                         # To format text safely for URL usage
import pyautogui                                            # To automate the 'Send' click event
from datetime import datetime                               # To capture real-time complaint timestamps
import hashlib                                              # To hash passwords securely

In [ ]:
try:                                                        # Start error handling for connection
    db = mysql.connector.connect(                           # Establish MySQL server connection
        host="localhost",                                   # Define server host address
        user="root",                                        # Define database username
        password="abcdefgh",                                # Define database password
        database="hostel_complaints"                        # Specify the database to use
    )                                                       # End connection call
    cursor = db.cursor()                                    # Create cursor object for SQL execution
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            id INT AUTO_INCREMENT PRIMARY KEY,
            email VARCHAR(255) UNIQUE NOT NULL,
            password_hash VARCHAR(64) NOT NULL,
            is_admin BOOLEAN DEFAULT FALSE,
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS complaints (
            id INT AUTO_INCREMENT PRIMARY KEY,
            complaint_no VARCHAR(20) UNIQUE NOT NULL,
            priority VARCHAR(50),
            email_id VARCHAR(255),
            category VARCHAR(255),
            subcategory VARCHAR(255),
            complaint_type VARCHAR(255),
            description TEXT,
            file_name VARCHAR(255),
            complaint_date DATETIME,
            status VARCHAR(50) DEFAULT 'New',
            room_number VARCHAR(50),
            student_name VARCHAR(255),
            hostel_block VARCHAR(50)
        )
    ''')
    try:
        cursor.execute("ALTER TABLE complaints ADD COLUMN status VARCHAR(50) DEFAULT 'New'")
    except Exception:
        pass
    try:
        cursor.execute("ALTER TABLE users ADD COLUMN is_admin BOOLEAN DEFAULT FALSE")
    except Exception:
        pass  
    try:
        cursor.execute("ALTER TABLE complaints ADD COLUMN room_number VARCHAR(50)")
        cursor.execute("ALTER TABLE complaints ADD COLUMN student_name VARCHAR(255)")
        cursor.execute("ALTER TABLE complaints ADD COLUMN hostel_block VARCHAR(50)")
    except Exception:
        pass
except Exception as e:                                      # Catch any connection errors
    print(f"Database Connection Error: {e}")                # Print error message to console


In [ ]:
priority_options = ["Select an Option", "Low", "Medium", "High"] # List of priority choices
support_categories = ["Select an Option", "Maintenance", "IT", "Housekeeping", "Food"] # List of main categories

sub_category_map = {                                        # Dictionary for dependent dropdowns
    "Maintenance": ["Select an Option", "Carpentry", "Civil", "Electrical", "Plumbing"], # Options for Maintenance
    "IT": ["Select an Option", "Application Issue", "IT", "Network Speed", "Others", "Password Issue", "Wifi"], # Options for IT
    "Housekeeping": ["Select an Option", "Laundry", "Pest Control", "Room Cleaning"], # Options for Housekeeping
    "Food": ["Select an Option", "Mess"]                    # Options for Food category
}                                                           # End of mapping dictionary

def update_subcategories(selected_category):                       # Support category change handler for dependent dropdowns
    options = sub_category_map.get(selected_category, ["Select an Option"])  # Lookup dependent subcategory options
    return gr.update(choices=options, value="Select an Option")            # Return updated dropdown config

complaint_cases = [                                         # List of specific complaint types
    "Select an Option", "AC Not Working", "App Is Not Working", "Bulb Is Not Working", "Chair Damaged",
    "Colour Fade", "Cot Damaged", "Door Stopper Is Not Working", "Electrical", "Drinking Water",
    "EWC Leakage", "Fan Not Working", "Flush Not Working", "Getting Stuck", "Hot Water",
    "Items Not Available", "Item Stuck", "Mirror Broken", "Missing", "No Water", "Pipe Broken",
    "Plug point issue", "Power not Available", "Study table Damaged", "Switch Is Not Working",
    "Tap Not Working", "Washing Machine Not Working", "Window Curtain Damaged", "Window Glass Broken",
    "Window Handle Broken", "Window Hinges Loose", "Others" # Closing specific cases
]                                                           # End of complaint list

In [ ]:
def generate_complaint_number():                                 # Function to auto-generate sequential complaint IDs
    try:                                                    # Start error handling for query
        cursor.execute("SELECT MAX(CAST(SUBSTRING(complaint_no, 5) AS UNSIGNED)) FROM complaints WHERE complaint_no LIKE 'CMP-%'")  # Get highest complaint number
        result = cursor.fetchone()                          # Fetch the result
        max_num = result[0] if result[0] else 0             # Extract number or default to 0
        new_num = max_num + 1                               # Increment by 1
        return f"CMP-{new_num:03d}"                          # Return formatted complaint number
    except Exception as e:                                  # Catch query errors
        return f"CMP-{datetime.now().strftime('%Y%m%d%H%M%S')}"  # Fallback to timestamp-based ID

def send_email_automation(email, complaint_no, category, subcategory, description): # Define automation function
    subject = f"Hostel Complaint Registered - {complaint_no}" # Create the email subject line
    current_date = datetime.now().strftime("%d-%m-%Y %H:%M:%S") # Format the current date/time
    
    body = f"""Dear Student,\n\nYour complaint has been successfully registered.\n\nComplaint Number : {complaint_no}\nCategory : {category}\nSub Category : {subcategory}\nComplaint Date : {current_date}\n\nDescription :\n{description}\n\nThank you,\nHostel Support Team""" # Build the body text
    
    encoded_subject = urllib.parse.quote(subject)           # Convert subject to URL-friendly format
    encoded_body = urllib.parse.quote(body)                 # Convert body to URL-friendly format
    
    gmail_url = f"https://mail.google.com/mail/?view=cm&fs=1&to={email}&su={encoded_subject}&body={encoded_body}" # Construct pre-filled Gmail URL
    webbrowser.open(gmail_url)                              # Open URL in default browser
    
    time.sleep(10)                                          # Wait 10 seconds for page to load
    pyautogui.hotkey('ctrl', 'enter')                       # Execute the "Send" click via keyboard hotkey

In [ ]:
def fetch_complaints(email):                                     # Function to retrieve all logs
    try:                                                    # Start error handling for query
        query = "SELECT complaint_no, category, priority, subcategory, room_number, student_name, hostel_block, complaint_date FROM complaints WHERE email_id = %s ORDER BY complaint_date DESC" # SQL to get data
        cursor.execute(query, (email,))                               # Run the query
        records = cursor.fetchall()                         # Store all results
        columns = ["Case Number", "Support Category", "Priority", "Hostel Sub Category", "Room Number", "Student Name", "Hostel Block", "Created Date"] # Define table headers
        return pd.DataFrame(records, columns=columns)       # Return results as a Pandas DataFrame
    except Exception as e:                                  # Catch SQL errors
        return pd.DataFrame({"Error": [str(e)]})            # Return error as a table entry


In [ ]:
def save_complaint(priority, email_id, category, subcategory, description, complaint, file, room_number, student_name, hostel_block): # Main save function
    if any(x == "Select an Option" for x in [priority, category, subcategory, complaint]) or not email_id.strip() or not room_number.strip() or not student_name.strip() or not hostel_block.strip(): # Check for empty inputs
        return "Please fill all required fields."           # Stop and return error message

    try:                                                    # Start database transaction
        complaint_no = generate_complaint_number()          # Generate next ID
        complaint_date = datetime.now()                     # Get current timestamp
        file_name = file.name if file else None             # Get filename if attachment exists

        sql = """INSERT INTO complaints (complaint_no, priority, email_id, category, subcategory, complaint_type, description, file_name, complaint_date, room_number, student_name, hostel_block) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)""" # Insert query
        values = (complaint_no, priority, email_id, category, subcategory, complaint, description, file_name, complaint_date, room_number, student_name, hostel_block) # Group values
        
        cursor.execute(sql, values)                         # Send data to MySQL
        db.commit()                                         # Save database changes

        send_email_automation(email_id, complaint_no, category, subcategory, description) # Start the email automation

        return f"Complaint Saved & Email Sent! No: {complaint_no}" # Return success message
    
    except Exception as e:                                  # Catch saving errors
        return f"Database Error: {str(e)}"                  # Return the error message

In [ ]:
def register_user(email, password, confirm_password):                                # Define function to handle new user account creation
    if not email or not password or not confirm_password:                            # Check if any registration form fields are empty
        return "Please fill all fields."                                             # Return error message for missing fields
    if password != confirm_password:                                                 # Ensure password matches the confirmation password
        return "Passwords do not match."                                             # Return error message for mismatched passwords
    
    hashed_password = hashlib.sha256(password.encode()).hexdigest()                  # Hash the plain text password securely using SHA-256
    try:                                                                             # Start secure database transaction block
        cursor.execute("INSERT INTO users (email, password_hash) VALUES (%s, %s)", (email, hashed_password))  # Insert new user credentials into database
        db.commit()                                                                  # Save database changes permanently
        return "Account created successfully! Please log in."                        # Return success message upon creation
    except mysql.connector.IntegrityError:                                           # Catch duplicate email database constraint errors
        return "Email already registered."                                           # Return user-friendly duplicate account error
    except Exception as e:                                                           # Catch any unexpected generic database errors
        return f"Database Error: {e}"                                                # Return backend error message for debugging

def login_user(email, password):                                                     # Define function to authenticate existing users
    if not email or not password:                                                    # Check if any registration form fields are empty
        return "Please enter email and password.", gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), None, "", pd.DataFrame(), pd.DataFrame()  # Return error if login fields are blank
        
    hashed_password = hashlib.sha256(password.encode()).hexdigest()                  # Hash the plain text password securely using SHA-256
    try:                                                                             # Start secure database transaction block
        cursor.execute("SELECT id, is_admin FROM users WHERE email = %s AND password_hash = %s", (email, hashed_password))  # Query database for matching email and password
        user = cursor.fetchone()                                                     # Retrieve user record if authentication succeeded
        if user:                                                                     # If user exists and password is correct
            is_admin = bool(user[1])                                                 # Check if user has database administrator rights
            if email.lower() == 'admin@hostel.com':                                  # Override admin status strictly for admin@hostel.com
                is_admin = True                                                      # Set admin authentication flag
            
            if is_admin:                                                             # Route to Admin Dashboard if admin flag is active
                admin_data = fetch_all_complaints()
                return f"Admin Login successful! Welcome, {email}", gr.update(visible=False), gr.update(visible=False), gr.update(visible=True), email, email, pd.DataFrame(), admin_data  # Return UI updates, email state, and admin table data
            else:                                                                    # Route to Standard User Dashboard if regular student
                user_data = fetch_complaints(email)
                return f"Login successful! Welcome, {email}", gr.update(visible=False), gr.update(visible=True), gr.update(visible=False), email, email, user_data, pd.DataFrame()  # Return UI updates, email state, and user complaint data
        else:                                                                        # Route to Standard User Dashboard if regular student
            return "Invalid email or password.", gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), None, "", pd.DataFrame(), pd.DataFrame()  # Return error message if credentials failed
    except Exception as e:                                                           # Catch any unexpected generic database errors
        return f"Database Error: {e}", gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), None, "", pd.DataFrame(), pd.DataFrame()  # Return backend error message for debugging

def logout():                                                                        # Define function to securely terminate user session
    return "Logged out.", gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), None  # Backend logic execution

def fetch_complaints(email):                                     # Function to retrieve all logs
    try:                                                    # Start error handling for query
        query = "SELECT complaint_no, category, priority, subcategory, room_number, student_name, hostel_block, complaint_date FROM complaints WHERE email_id = %s ORDER BY complaint_date DESC" # SQL to get data
        cursor.execute(query, (email,))                               # Run the query
        records = cursor.fetchall()                         # Store all results
        columns = ["Case Number", "Support Category", "Priority", "Hostel Sub Category", "Room Number", "Student Name", "Hostel Block", "Created Date"] # Define table headers
        return pd.DataFrame(records, columns=columns)       # Return results as a Pandas DataFrame
    except Exception as e:                                  # Catch SQL errors
        return pd.DataFrame({"Error": [str(e)]})            # Return error as a table entry


def fetch_all_complaints():                                                          # Define function to retrieve all database complaints
    try:                                                                             # Start secure database transaction block
        query = "SELECT complaint_no, email_id, category, priority, room_number, student_name, hostel_block, status, complaint_date FROM complaints ORDER BY complaint_date DESC"  # SQL command to get all complaints unconditionally
        cursor.execute(query)                                                        # Execute the unrestricted fetch query
        records = cursor.fetchall()                                                  # Extract all available system rows
        columns = ["Complaint No", "User Email", "Category", "Priority", "Room Number", "Student Name", "Hostel Block", "Status", "Created Date"]  # Define the UI dataframe table column headers
        return pd.DataFrame(records, columns=columns)                                # Return formatted pandas dataframe for Admin UI
    except Exception as e:                                                           # Catch any unexpected generic database errors
        return pd.DataFrame({"Error": [str(e)]})                                     # Return formatted pandas dataframe for Admin UI


def fetch_complaint_details(complaint_no):                                           # Define function to get specific case fields
    if not complaint_no or complaint_no == "Select Complaint":                       # Check if valid complaint ID passed
        return "", "", "", "", "Select an Option", "Select an Option", "Select an Option", "Select an Option", "", "New"  # Return empty defaults if no ID is loaded
        
    try:                                                                             # Start secure database transaction block
        query = "SELECT email_id, room_number, student_name, hostel_block, priority, category, subcategory, complaint_type, description, status FROM complaints WHERE complaint_no = %s"  # SQL command to get detailed fields for single case
        cursor.execute(query, (complaint_no,))                                       # Execute targeted fetch using Complaint ID
        rec = cursor.fetchone()                                                      # Retrieve the singular row result
        if rec:                                                                      # Check if the database row exists
            return rec[0], rec[1], rec[2], rec[3], rec[4], rec[5], rec[6], rec[7], rec[8], (rec[9] if rec[9] else 'New')  # Return array of fields to bind to UI components
        return "", "", "", "", "Select an Option", "Select an Option", "Select an Option", "Select an Option", "", "New"  # Return empty defaults if no ID is loaded
    except Exception as e:                                                           # Catch any unexpected generic database errors
        import traceback                                                             # Import traceback to capture exact Python exceptions
        with open("fetch_error_log.txt", "w") as f:                                  # Open an error log file for debugging
            f.write(traceback.format_exc())                                          # Write the exact exception stack trace to file
        return "Error", "Error", "Error", "Error", "Error", "Error", "Error", "Error", str(e), "Error"  # Return safe fallback variables if backend panics


def fetch_oldest_complaint_by_email(email):                                          # Define automation to extract user's oldest case
    if not email:                                                                    # Backend logic execution
        return "Select Complaint"                                                    # Backend logic execution
    try:                                                                             # Start secure database transaction block
        query = "SELECT complaint_no FROM complaints WHERE email_id = %s ORDER BY complaint_date ASC LIMIT 1"  # SQL command to get all complaints unconditionally
        cursor.execute(query, (email,))                                              # Execute targeted fetch using Complaint ID
        rec = cursor.fetchone()                                                      # Retrieve the singular row result
        if rec:                                                                      # Check if the database row exists
            return rec[0]                                                            # Return array of fields to bind to UI components
        return "Select Complaint"                                                    # Backend logic execution
    except Exception as e:                                                           # Catch any unexpected generic database errors
        return "Select Complaint"                                                    # Backend logic execution


def get_all_complaint_numbers():                                                     # Function to populate dropdown with all cases
    try:                                                                             # Start secure database transaction block
        cursor.execute("SELECT complaint_no FROM complaints ORDER BY complaint_date DESC")  # Query database for matching email and password
        nums = [row[0] for row in cursor.fetchall()]                                 # Format MySQL tuple results into a clean list
        return gr.Dropdown(choices=["Select Complaint"] + nums)                      # Backend logic execution
    except Exception as e:                                                           # Catch any unexpected generic database errors
        return gr.Dropdown(choices=["Select Complaint"])                             # Backend logic execution


def send_status_update_email(complaint_no, new_status, description):                 # Function to automate status change emails
    try:                                                                             # Start secure database transaction block
        cursor.execute("SELECT email_id FROM complaints WHERE complaint_no = %s", (complaint_no,))  # Query database for matching email and password
        row = cursor.fetchone()                                                      # Get the student email record
        if not row:                                                                  # Backend logic execution
            return                                                                   # Backend logic execution
        
        email = row[0]                                                               # Extract email string from tuple
        subject = f"Hostel Complaint {complaint_no} Status Update: {new_status}"     # Format email notification subject line
        body = f"Dear Student,\n\nThe status of your complaint ({complaint_no}) has been updated to: {new_status}.\n\nAdmin Remarks:\n{description}\n\nThank you,\nHostel Support Team"  # Format the email body with admin remarks
        
        encoded_subject = urllib.parse.quote(subject)                                # Backend logic execution
        encoded_body = urllib.parse.quote(body)                                      # Backend logic execution
        
        gmail_url = f"https://mail.google.com/mail/?view=cm&fs=1&to={email}&su={encoded_subject}&body={encoded_body}"  # Backend logic execution
        webbrowser.open(gmail_url)                                                   # Backend logic execution
        time.sleep(10)                                                               # Backend logic execution
        pyautogui.hotkey('ctrl', 'enter')                                            # Backend logic execution
    except Exception as e:                                                           # Catch any unexpected generic database errors
        print(f"Email failure: {e}")                                                 # Print local console failure if Gmail automation crashes

def update_complaint_status(complaint_no, priority, category, subcategory, complaint_type, description, status):  # Function to update existing case data
    if not complaint_no or complaint_no == "Select Complaint":                       # Check if valid complaint ID passed
        return "Please select a valid complaint."                                    # Abort update if valid case isn't targeted
    try:                                                                             # Start secure database transaction block
        sql = "UPDATE complaints SET priority=%s, category=%s, subcategory=%s, complaint_type=%s, description=%s, status=%s WHERE complaint_no=%s"  # SQL command to modify row fields
        cursor.execute(sql, (priority, category, subcategory, complaint_type, description, status, complaint_no))  # Backend logic execution
        db.commit()                                                                  # Save database changes permanently
        
        send_status_update_email(complaint_no, status, description)                  # Trigger Gmail notification after DB updates
        
        return f"Complaint {complaint_no} updated successfully! Notification prepared."  # Return success to UI update status box
    except Exception as e:                                                           # Catch any unexpected generic database errors
        return f"Database Error: {e}"                                                # Return backend error message for debugging

def on_select_complaint(evt: gr.SelectData, df):                                     # Function to bridge table clicks to dropdown
    if getattr(df, 'iloc', None) is not None:                                        # Check dataframe type for Gradio selection event
        return df.iloc[evt.index[0], 0]                                              # Extract complaint string from dataframe index
    elif isinstance(df, dict) and 'data' in df:                                      # Fallback dictionary extraction for old Gradio
        return df['data'][evt.index[0]][0]                                           # Extract complaint from Gradio dictionary format
    else:                                                                            # Fallback direct array extraction
        return df[evt.index[0]][0]                                                   # Return complaint string for simple list/dict formats

In [ ]:
with gr.Blocks(title="Hostel Complaint Management System", theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate", font=[gr.themes.GoogleFont("Outfit"), "sans-serif"])) as app:  # Initialize main Gradio visual application block
    logged_in_user = gr.State(value=None)                                            # Create persistent browser session state for Auth

    with gr.Column(visible=True) as auth_view:
        gr.Markdown("<h1 style='text-align: center; margin-bottom: 1rem'>Hostel Complaint Management</h1>")                                       # Create Authentication view wrapper (default active)
        with gr.Tabs():                                                              # Initialize tabbed navigation structure
            with gr.Tab("🔐 Login"):                                                  # Create Login tab interface
                login_email = gr.Textbox(label="Email")                              # Input box for user email address
                login_password = gr.Textbox(label="Password", type="password")       # Input box for hidden user password
                login_btn = gr.Button("Login", variant="primary")                    # Submit button for Login validation
                login_status = gr.Textbox(label="Status")                            # Text box to display Login success/failure
            with gr.Tab("📝 Sign Up"):                                                # Create Registration tab interface
                reg_email = gr.Textbox(label="Email")                                # Input box for new email address
                reg_password = gr.Textbox(label="Password", type="password")         # Input box for new hidden password
                reg_confirm = gr.Textbox(label="Confirm Password", type="password")  # Input box to confirm new password
                reg_btn = gr.Button("Sign Up", variant="primary")                    # Submit button to create account
                reg_status = gr.Textbox(label="Status")                              # Text box to output registration results
                        
    with gr.Column(visible=False) as admin_view:                                     # Create Admin view wrapper (default hidden)
        admin_logout_btn = gr.Button("Admin Logout", size="sm")                      # Button for Admins to log out safely
        admin_logout_status = gr.Textbox(label="Logout Status", visible=False)       # Hidden layout element for logout returns
        with gr.Tabs():                                                              # Initialize tabbed navigation structure
            with gr.Tab("📊 Admin Dashboard"):                                        # Create primary Admin visualization tab
                admin_refresh_btn = gr.Button("Refresh Data")                        # Button to pull latest cases from MySQL
                admin_table = gr.Dataframe(interactive=False)                        # Non-editable table containing all raw complaints
                admin_refresh_btn.click(fn=fetch_all_complaints, outputs=admin_table)  # Bind table refresh function to the button
            with gr.Tab("✏️ Update Case Status"):                                    # Create interactive Admin update tab
                gr.Markdown("---")                                                   # Render horizontal visual separation line
                gr.Markdown("### Update Selected Case Below")
                with gr.Row():                                                       # Group components horizontally in UI
                    admin_complaint_drop = gr.Dropdown(choices=["Select Complaint"], label="Selected Case ID", interactive=True, visible=True, allow_custom_value=True)  # Dropdown array containing complaint IDs to update
                with gr.Row():                                                       # Group components horizontally in UI
                    admin_user_email = gr.Textbox(label="User Email ID", interactive=False)  # Display selected complaint's email
                    admin_room_number = gr.Textbox(label="Room Number", interactive=False)  # Display selected complaint's room number
                with gr.Row():                                                       # Group components horizontally in UI
                    admin_student_name = gr.Textbox(label="Student Name", interactive=False)  # Display selected complaint's student name
                    admin_hostel_block = gr.Textbox(label="Hostel Block", interactive=False)  # Display selected complaint's hostel block
                with gr.Row():                                                       # Group components horizontally in UI
                    admin_status = gr.Dropdown(choices=["New", "In Progress", "Resolved", "Closed", "Reopened"], value="New", label="Status", allow_custom_value=True)  # Dropdown mapping New to Reopened cases
                    admin_priority = gr.Dropdown(choices=priority_options, label="Priority", allow_custom_value=True)  # Dropdown representing priority level
                with gr.Row():                                                       # Group components horizontally in UI
                    admin_category = gr.Dropdown(choices=support_categories, label="Support Category", allow_custom_value=True)  # Dropdown identifying the broad support category
                    admin_subcategory = gr.Dropdown(choices=["Select an Option"], label="Hostel Sub Category", allow_custom_value=True)  # Dropdown identifying exact support category
                admin_case = gr.Dropdown(choices=complaint_cases, label="Support Case Complaints", allow_custom_value=True)  # Dropdown for detailed specific case selections
                admin_desc = gr.Textbox(label="Admin Remarks / Description", lines=5)  # Textbox for Admin internal/external remarks
                admin_update_btn = gr.Button("Update Case & Notify User", variant="primary")  # Button to execute MySQL update and fire email
                admin_update_status = gr.Textbox(label="Update Confirmation")        # Box rendering post-update success notes
                        
                admin_table.select(fn=on_select_complaint, inputs=[admin_table], outputs=[admin_complaint_drop])  # Enable clickable table rows to push ID to dropdown
                admin_category.change(fn=update_subcategories, inputs=admin_category, outputs=admin_subcategory)  # Automatically change allowed subcategories when category changes
                admin_complaint_drop.change(fn=fetch_complaint_details, inputs=admin_complaint_drop, outputs=[admin_user_email, admin_room_number, admin_student_name, admin_hostel_block, admin_priority, admin_category, admin_subcategory, admin_case, admin_desc, admin_status])  # Fetch array of SQL fields when Dropdown value changes
                admin_update_btn.click(fn=update_complaint_status, inputs=[admin_complaint_drop, admin_priority, admin_category, admin_subcategory, admin_case, admin_desc, admin_status], outputs=admin_update_status)  # Execute Update function sending all fields to DB

    with gr.Column(visible=False) as app_view:                                       # Create Student view wrapper (default hidden)
        logout_btn = gr.Button("Logout", size="sm")                                  # Button for Students to log out securely
        logout_status = gr.Textbox(label="Logout Status", visible=False)             # Hidden box for student logout routing
        with gr.Tabs():                                                              # Initialize tabbed navigation structure
            with gr.Tab("📋 Register Complaint"):                                     # Create new complaint initiation tab
                gr.Markdown("## Hostel Support Request Details")
                with gr.Row():                                                       # Group components horizontally in UI
                    priority_drop = gr.Dropdown(choices=priority_options, value="Select an Option", label="Priority")  # Priority selector for student issue urgency
                    email_text = gr.Textbox(interactive=False, label="User Email ID", placeholder="Enter your university email")  # Locked pre-filled email bounding input
                    room_number_text = gr.Textbox(label="Room Number")              # Input for student's room number
                with gr.Row():                                                       # Group components horizontally in UI
                    student_name_text = gr.Textbox(label="Student Name")            # Input for student's name
                    hostel_block_text = gr.Textbox(label="Hostel Block")            # Input for hostel block
                    category_drop = gr.Dropdown(choices=support_categories, value="Select an Option", label="Support Category")  # Main category UI target for students
                subcategory_drop = gr.Dropdown(choices=["Select an Option"], value="Select an Option", label="Hostel Sub Category")  # Main category UI target for students
                description_text = gr.Textbox(label="Description", lines=4, placeholder="Describe your issue...")  # Input area for lengthy specific explanations
                complaints_drop = gr.Dropdown(choices=complaint_cases, value="Select an Option", label="Support Case Complaints")  # Detailed option matrix for granular issue tracking
                file_upload = gr.File(label="Upload Document")                       # Gradio generic file upload parameter (Optional)
                output_status = gr.Textbox(label="Status")                           # Textblock rendering UI success creation note
                with gr.Row():                                                       # Group components horizontally in UI
                    save_btn = gr.Button("Save", variant="primary")                  # Key execution button for DB insertion
                    close_btn = gr.Button("Close")                                   # Button providing abort/close routing
                category_drop.change(fn=update_subcategories, inputs=category_drop, outputs=subcategory_drop)  # Cascade logic to dynamically block subcategories
                save_btn.click(fn=save_complaint, inputs=[priority_drop, email_text, category_drop, subcategory_drop, description_text, complaints_drop, file_upload, room_number_text, student_name_text, hostel_block_text], outputs=output_status)  # Bind DB save logic and email triggering to UI
                close_btn.click(fn=lambda: "Form Closed", outputs=output_status)     # Bind dummy close payload directly to output box
            with gr.Tab("📂 View Complaints"):                                        # Create personal complaints viewer tab
                gr.Markdown("## My Complaint Records")
                complaint_table = gr.Dataframe(interactive=False)                    # Non-interactive element to view own cases
                refresh_btn = gr.Button("Refresh Complaints")                        # Manual update button for user table
                refresh_btn.click(fn=fetch_complaints, inputs=logged_in_user, outputs=complaint_table)  # Bind MySQL isolation lookup mapped strictly to User Email

    reg_btn.click(fn=register_user, inputs=[reg_email, reg_password, reg_confirm], outputs=reg_status)  # Bind DB signup execution algorithm to Register button
    login_btn.click(                                                                 # Bind authentication lookup returning UI structural visibility
        fn=login_user,                                                               # Gradio component configuration
        inputs=[login_email, login_password],                                        # Gradio component configuration
        outputs=[login_status, auth_view, app_view, admin_view, logged_in_user, email_text, complaint_table, admin_table]      # Gradio component configuration
    )                                                                                # Gradio component configuration
    for btn in [logout_btn, admin_logout_btn]:                                       # Iterate binding logout logic to both app modes
        btn.click(                                                                   # Bind UI logic resetting Auth_View routing
            fn=logout,                                                               # Gradio component configuration
            inputs=None,                                                             # Gradio component configuration
            outputs=[login_status, auth_view, app_view, admin_view, logged_in_user]  # Gradio component configuration
        ).then(                                                                      # Gradio component configuration
            fn=lambda: ("", "", "", ""), outputs=[login_email, login_password, login_status, reg_status]  # Wipe all inputs out securely to prevent session leaks
        )                                                                            # Gradio component configuration

In [ ]:
app.launch()                                                # Start the Gradio application